# Houseplant SFT v2 — Colab T4: обучение + оценка

Один ноутбук: установка → pilot → полное QLoRA на `data/sft_v2/sft_train.jsonl` (1 060 пар, отбор судьёй, см. `reports/sft_v2_data.md`) → адаптер на Drive → base vs SFT на plant + garden + general, судья, правило выбора чекпоинта, test, графики.

Стек (pinned): `transformers==4.57.1` + `trl==0.24.0` + `peft==0.17.1` + `bitsandbytes` (Linux).

1. Runtime → Change runtime type → **T4 GPU**.
2. Secrets (ключ слева) → `OPENROUTER_API_KEY`, включить Notebook access.
3. Runtime → **Run all**. Вкладку не закрывать.

Повторный Run all безопасен: если адаптер уже на Drive (`MyDrive/rk1_houseplant/s2_sft_v2`), обучение пропускается; оценка пишется на Drive построчно и продолжает с места остановки.

- `SMOKE = True` — оценка по 3 примера на набор в отдельную папку (обучение не меняется).
- `FORCE_TRAIN = True` — переобучить, даже если адаптер на Drive уже есть.
- Старые `eval_s1` / `eval_smoke` не переиспользовать: другой системный промпт, база там оценена по-старому.
- Если после старого `--force-reinstall` сломан numpy/torch: Runtime → Disconnect and delete runtime. Не используйте `pip install -U` / `--force-reinstall` на весь файл требований.

In [ ]:
from google.colab import drive, userdata
import os

SMOKE = False
FORCE_TRAIN = False

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rk1_houseplant'
ADAPTER = f'{DRIVE}/s2_sft_v2'
OUT = f'{DRIVE}/eval_smoke_s2' if SMOKE else f'{DRIVE}/eval_s2'
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
os.makedirs(OUT, exist_ok=True)
TRAIN = FORCE_TRAIN or not os.path.exists(f'{ADAPTER}/adapter_model.safetensors')
print('train:', TRAIN, '| adapter:', ADAPTER, '| out:', OUT)

In [ ]:
%cd /content
!git clone https://github.com/Schaft-s/LLM || (cd LLM && git pull)
%cd /content/LLM
import sys
sys.path.insert(0, '/content/LLM')
!wc -l data/sft_v2/sft_train.jsonl  # expect 1060
# Safe install: only our pins. Needs bitsandbytes>=0.49.2 for Colab CUDA 13.
!bash scripts/colab_install.sh
!python scripts/smoke_colab_stack.py
!git rev-parse HEAD > "{OUT}/git_commit.txt"
!mkdir -p "{OUT}/configs" && cp configs/*.yaml "{OUT}/configs/"
!nvidia-smi --query-gpu=name,memory.total --format=csv > "{OUT}/gpu.txt"

## Pilot (20 steps) → reload + 1 generation

In [ ]:
import subprocess

def run(cmd, log):
    # stream output to the cell and to a log on Drive; fail the cell on error
    with open(log, 'a') as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end=''); f.write(line)
    if p.wait():
        raise SystemExit(f'{cmd} failed, see {log}')

if TRAIN:
    run([sys.executable, 'scripts/train_qlora.py', '--pilot'], f'{OUT}/train_pilot.log')
    from src.train.generate import generate_chat, load_base_or_adapter
    from src.train.sft_qlora import load_yaml
    import gc, torch
    model_cfg = load_yaml('configs/model.yaml')
    model, tok = load_base_or_adapter(model_cfg, adapter_dir='artifacts/weights/pilot_qlora')
    print(generate_chat(
        model, tok,
        'My snake plant has mushy leaves at the base after weekly watering in low light. What should I check first?',
        system_prompt=model_cfg.get('system_prompt', '').strip(),
        max_new_tokens=256, temperature=0.2,
    ))
    del model; gc.collect(); torch.cuda.empty_cache()
else:
    print('adapter already on Drive → skip pilot')

## S2: полное QLoRA на SFT v2 → копия на Drive

In [ ]:
if TRAIN:
    run([sys.executable, 'scripts/train_qlora.py'], f'{OUT}/train_s2.log')
    !rm -rf "{ADAPTER}" && cp -r artifacts/weights/s2_sft_v2 "{ADAPTER}"
print(sorted(os.listdir(ADAPTER)))
assert os.path.exists(f'{ADAPTER}/adapter_model.safetensors')

## Оценка: base vs SFT

Генерация → автопроверка `no_invented_experience` → судья (OpenRouter) → правило выбора чекпоинта (судья по plant_dev ∪ garden_dev не хуже базы; general_dev не падает больше чем на 5 п.п.) → test → графики. Возобновляемо.

In [ ]:
LIMIT = '--limit 3' if SMOKE else ''
!python scripts/run_eval_pipeline.py --out-dir "{OUT}" --adapter-root "{ADAPTER}" {LIMIT} 2>&1 | tee -a "{OUT}/run.log"

In [ ]:
import glob
from IPython.display import Image, Markdown, display

display(Markdown(open(f'{OUT}/summary.md').read()))
for p in sorted(glob.glob(f'{OUT}/plots/*.png')):
    print(p)
    display(Image(p))
drive.flush_and_unmount()
print('Drive flushed')